# **TRANSFORMACIÓN DE LOS DATOS**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **INGRESOS HOSPITALARIOS**
En primer lugar, se renombraron las columnas como Comunidad Autonoma, Periodo e Ingresos_Hospitalarios. Posteriormente, se estandarizaron los nombres de las comunidades autónomas, eliminando aclaraciones entre paréntesis y unificando su denominación (por ejemplo, Asturias (Principado de) → Principado de Asturias). También se eliminaron las tildes para asegurar la consistencia de los nombres.

Además, el registro “Ceuta y Melilla” se separó en dos filas independientes (Ceuta y Melilla). Dado que la variable representa una tasa por 1000 habitantes, el valor se replicó para ambas ciudades en lugar de dividirse. Finalmente, se transformaron los tipos de datos para que Periodo fuese numérico y Ingresos_Hospitalarios se interpretara correctamente como valor decimal.

Ademas de la limpieza y homogeneizacion de nombres, se genero un registro adicional para Espana en cada periodo. Dado que la variable representa una tasa por 1000 habitantes, el valor nacional se calculo como la media de los valores de todas las comunidades autonomas para cada año.

In [3]:
import pandas as pd
import unicodedata

ruta = "/content/drive/MyDrive/DATOS_TFG/Ingresos_Hospitalarios.csv"

df = pd.read_csv(ruta, sep=";", encoding="latin1")

# Renombrar columnas
df.columns = ["Comunidad Autonoma", "Periodo", "Ingresos_Hospitalarios"]

# Limpiar espacios
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].astype(str).str.strip()

# Homogeneizar nombres
reemplazos = {
    "Asturias (Principado de)": "Principado de Asturias",
    "Balears (Illes)": "Islas Baleares",
    "Madrid (Comunidad de)": "Comunidad de Madrid",
    "Murcia (Región de)": "Region de Murcia",
    "Navarra (Comunidad Foral de)": "Navarra",
    "Rioja (La)": "La Rioja"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos)

# Separar Ceuta y Melilla en dos filas con el mismo valor
filas_nuevas = []

for _, row in df.iterrows():
    if row["Comunidad Autonoma"] == "Ceuta y Melilla":
        filas_nuevas.append({
            "Comunidad Autonoma": "Ceuta",
            "Periodo": row["Periodo"],
            "Ingresos_Hospitalarios": row["Ingresos_Hospitalarios"]
        })
        filas_nuevas.append({
            "Comunidad Autonoma": "Melilla",
            "Periodo": row["Periodo"],
            "Ingresos_Hospitalarios": row["Ingresos_Hospitalarios"]
        })
    else:
        filas_nuevas.append({
            "Comunidad Autonoma": row["Comunidad Autonoma"],
            "Periodo": row["Periodo"],
            "Ingresos_Hospitalarios": row["Ingresos_Hospitalarios"]
        })

df = pd.DataFrame(filas_nuevas)

# Funcion para quitar tildes
def quitar_tildes(texto):
    texto = str(texto)
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

# Quitar tildes
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].apply(quitar_tildes)

# Convertir Periodo a numerico
df["Periodo"] = pd.to_numeric(df["Periodo"], errors="coerce")

# Convertir Ingresos_Hospitalarios a numerico
df["Ingresos_Hospitalarios"] = (
    df["Ingresos_Hospitalarios"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)
df["Ingresos_Hospitalarios"] = pd.to_numeric(df["Ingresos_Hospitalarios"], errors="coerce")

# Crear fila de España como media de las comunidades en cada periodo
df_espana = (
    df.groupby("Periodo", as_index=False)["Ingresos_Hospitalarios"]
    .mean()
)

df_espana["Comunidad Autonoma"] = "Espana"
df_espana = df_espana[["Comunidad Autonoma", "Periodo", "Ingresos_Hospitalarios"]]

# Unir al dataframe original
df = pd.concat([df, df_espana], ignore_index=True)

# Ordenar
df = df.sort_values(["Periodo", "Comunidad Autonoma"]).reset_index(drop=True)

In [4]:
ruta_salida = "/content/drive/MyDrive/DATOS_TFG_LIMPIOS/Ingresos_Hospitalarios.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

## **TRATAMIENTO DE AGUAS**
Se realizó un proceso de limpieza y transformación del dataset de tratamiento de aguas para garantizar la consistencia de los datos. En primer lugar, se filtró el conjunto de datos para conservar únicamente el indicador “Volumen de aguas residuales tratadas”, eliminando posteriormente la columna correspondiente al tipo de indicador al no ser necesaria para el análisis.

Posteriormente, se homogeneizaron los nombres de las comunidades autónomas para mantener la misma nomenclatura que en el resto de datasets, eliminando además las tildes para asegurar la consistencia textual. Asimismo, el registro agregado “Ceuta y Melilla” se separó en dos filas independientes (Ceuta y Melilla), repartiendo el volumen total de aguas tratadas entre ambas ciudades autónomas.

Finalmente, se transformaron las variables Periodo y Tratamiento_Aguas a formato numérico, corrigiendo los separadores de miles y decimales para permitir su correcta interpretación y posterior análisis.

In [5]:
import pandas as pd
import unicodedata

ruta = "/content/drive/MyDrive/DATOS_TFG/Tratamiento_Aguas.csv"

df = pd.read_csv(ruta, sep=";", encoding="utf-8")

# Renombrar columnas
df.columns = ["Comunidad Autonoma", "Tipo_Indicador", "Periodo", "Volumen_Agua_Tratada"]

# Limpiar espacios
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].astype(str).str.strip()
df["Tipo_Indicador"] = df["Tipo_Indicador"].astype(str).str.strip()

# Filtrar solo el indicador deseado
df = df[df["Tipo_Indicador"] == "Volumen de aguas residuales tratadas"]

# Eliminar la columna del indicador
df = df.drop(columns=["Tipo_Indicador"])

# Homogeneizar nombres de comunidades
reemplazos = {
    "Asturias, Principado de": "Principado de Asturias",
    "Balears, Illes": "Islas Baleares",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia, Región de": "Region de Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja, La": "La Rioja",
    "Comunitat Valenciana": "Comunidad Valenciana"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos)

# Convertir a numerico (quitar separadores de miles y arreglar decimales)
df["Volumen_Agua_Tratada"] = (
    df["Volumen_Agua_Tratada"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["Volumen_Agua_Tratada"] = pd.to_numeric(df["Volumen_Agua_Tratada"], errors="coerce")

# Separar Ceuta y Melilla en dos filas dividiendo el valor
filas_nuevas = []

for _, row in df.iterrows():
    if row["Comunidad Autonoma"] == "Ceuta y Melilla":
        valor = row["Volumen_Agua_Tratada"] / 2

        filas_nuevas.append({
            "Comunidad Autonoma": "Ceuta",
            "Periodo": row["Periodo"],
            "Volumen_Agua_Tratada": valor
        })

        filas_nuevas.append({
            "Comunidad Autonoma": "Melilla",
            "Periodo": row["Periodo"],
            "Volumen_Agua_Tratada": valor
        })
    else:
        filas_nuevas.append({
            "Comunidad Autonoma": row["Comunidad Autonoma"],
            "Periodo": row["Periodo"],
            "Volumen_Agua_Tratada": row["Volumen_Agua_Tratada"]
        })

df = pd.DataFrame(filas_nuevas)

# Funcion para quitar tildes
def quitar_tildes(texto):
    texto = str(texto)
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].apply(quitar_tildes)

# Convertir periodo a numerico
df["Periodo"] = pd.to_numeric(df["Periodo"], errors="coerce")

In [6]:
ruta_salida = "/content/drive/MyDrive/DATOS_TFG_LIMPIOS/Volumen_Aguas_Tratadas.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

## **RENTA ANUAL MEDIA**
Se realizó la limpieza del dataset de renta anual renombrando las columnas y filtrando únicamente el indicador de renta neta media por hogar. Después, se eliminó la columna del indicador, se transformó Total Nacional en Espana, se eliminaron los códigos numéricos iniciales y se homogeneizaron los nombres de las comunidades autónomas. Además, se aplicó una normalización del texto para corregir problemas de codificación y eliminar tildes, de forma que nombres como AndalucA­a pasaran a Andalucia. Por último, en caso de aparecer Ceuta y Melilla, el registro se separó en dos filas, y las variables Periodo y Renta_Anual se convirtieron a formato numérico.

In [7]:
import pandas as pd
import unicodedata
import re

ruta = "/content/drive/MyDrive/DATOS_TFG/Renta_anual.csv"

df = pd.read_csv(ruta, sep=";", encoding="latin1")

# Renombrar columnas
df.columns = ["Comunidad Autonoma", "Indicador", "Periodo", "Renta_Anual"]

# Filtrar solo el indicador deseado
df = df[df["Indicador"].str.contains("Renta neta media por hogar", case=False, na=False)]

# Eliminar columna del indicador
df = df.drop(columns=["Indicador"])

# Limpiar espacios iniciales/finales
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].astype(str).str.strip()

# Cambiar Total Nacional por Espana
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace({
    "Total Nacional": "Espana"
})

# Quitar codigos numericos iniciales
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].str.replace(r"^\d+\s*", "", regex=True)

# Corregir problemas frecuentes de codificacion
reemplazos_codificacion = {
    "AndalucA­a": "Andalucia",
    "AragA3n": "Aragon",
    "CataluA±a": "Cataluna",
    "Castilla y LeA3n": "Castilla y Leon",
    "PaA­s Vasco": "Pais Vasco",
    "Murcia, RegiA3n de": "Region de Murcia",
    "EspaA±a": "Espana"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos_codificacion)

# Homogeneizar nombres
reemplazos = {
    "Asturias (Principado de)": "Principado de Asturias",
    "Asturias, Principado de": "Principado de Asturias",
    "Balears (Illes)": "Islas Baleares",
    "Balears, Illes": "Islas Baleares",
    "Madrid (Comunidad de)": "Comunidad de Madrid",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia (Región de)": "Region de Murcia",
    "Murcia, Región de": "Region de Murcia",
    "Navarra (Comunidad Foral de)": "Navarra",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja (La)": "La Rioja",
    "Rioja, La": "La Rioja",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Valenciana, Comunidad": "Comunidad Valenciana"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos)

# Funcion para limpiar texto, quitar tildes y caracteres raros
def limpiar_texto(texto):
    texto = str(texto)

    # Arreglar algunos caracteres raros tipicos
    texto = texto.replace("Ã¡", "a").replace("Ã©", "e").replace("Ã­", "i")
    texto = texto.replace("Ã³", "o").replace("Ãº", "u").replace("Ã±", "n")
    texto = texto.replace("Â", "").replace("Ã", "")

    # Normalizar y quitar tildes
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    # Limpiar espacios repetidos
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].apply(limpiar_texto)

# Separar Ceuta y Melilla solo si aparece junto
filas_nuevas = []

for _, row in df.iterrows():
    if row["Comunidad Autonoma"] == "Ceuta y Melilla":
        valor = pd.to_numeric(
            str(row["Renta_Anual"]).replace(".", "").replace(",", "."),
            errors="coerce"
        )
        filas_nuevas.append(["Ceuta", row["Periodo"], valor / 2])
        filas_nuevas.append(["Melilla", row["Periodo"], valor / 2])
    else:
        filas_nuevas.append([row["Comunidad Autonoma"], row["Periodo"], row["Renta_Anual"]])

df = pd.DataFrame(filas_nuevas, columns=["Comunidad Autonoma", "Periodo", "Renta_Anual"])

# Convertir columnas a numerico
df["Periodo"] = pd.to_numeric(df["Periodo"], errors="coerce")

df["Renta_Anual"] = (
    df["Renta_Anual"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["Renta_Anual"] = pd.to_numeric(df["Renta_Anual"], errors="coerce")

# Eliminar filas vacias si las hubiera
df = df.dropna(subset=["Comunidad Autonoma", "Periodo", "Renta_Anual"])

In [8]:
ruta_salida = "/content/drive/MyDrive/DATOS_TFG_LIMPIOS/Renta_anual.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

## **RESIDUOS RECOGIDOS**
Se realizó la limpieza del dataset de residuos renombrando las columnas y filtrando únicamente el indicador de interés. Posteriormente, se eliminó la columna del indicador, se transformó Total Nacional en Espana, se eliminaron los códigos numéricos iniciales y se homogeneizaron los nombres de las comunidades autónomas. Además, se corrigieron problemas de codificación y se eliminaron las tildes para asegurar la consistencia textual. En caso de aparecer Ceuta y Melilla, el registro se separó en dos filas. Finalmente, las variables Periodo y Residuos se convirtieron a formato numérico para facilitar su análisis.

In [9]:
import pandas as pd
import unicodedata
import re

ruta = "/content/drive/MyDrive/DATOS_TFG/Residuos.csv"

df = pd.read_csv(ruta, sep=";", encoding="latin1")

# Renombrar columnas
df.columns = ["Comunidad Autonoma", "Indicador", "Periodo", "Residuos"]

# Filtrar solo el indicador que quieres conservar
# Si luego ves que el texto exacto cambia, ajustamos esta línea
df = df[df["Indicador"].str.contains("residuos", case=False, na=False)]

# Eliminar columna del indicador
df = df.drop(columns=["Indicador"])

# Limpiar espacios
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].astype(str).str.strip()

# Cambiar Total Nacional por Espana
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace({
    "Total Nacional": "Espana"
})

# Quitar codigos numericos iniciales
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].str.replace(r"^\d+\s*", "", regex=True)

# Corregir problemas frecuentes de codificacion
reemplazos_codificacion = {
    "AndalucA­a": "Andalucia",
    "AragA3n": "Aragon",
    "CataluA±a": "Cataluna",
    "Castilla y LeA3n": "Castilla y Leon",
    "PaA­s Vasco": "Pais Vasco",
    "Murcia, RegiA3n de": "Region de Murcia",
    "EspaA±a": "Espana"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos_codificacion)

# Homogeneizar nombres
reemplazos = {
    "Asturias (Principado de)": "Principado de Asturias",
    "Asturias, Principado de": "Principado de Asturias",
    "Balears (Illes)": "Islas Baleares",
    "Balears, Illes": "Islas Baleares",
    "Madrid (Comunidad de)": "Comunidad de Madrid",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia (Región de)": "Region de Murcia",
    "Murcia, Región de": "Region de Murcia",
    "Navarra (Comunidad Foral de)": "Navarra",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja (La)": "La Rioja",
    "Rioja, La": "La Rioja",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Valenciana, Comunidad": "Comunidad Valenciana",
    "Total nacional": "España"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos)

# Funcion para limpiar texto y quitar tildes
def limpiar_texto(texto):
    texto = str(texto)

    texto = texto.replace("Ã¡", "a").replace("Ã©", "e").replace("Ã­", "i")
    texto = texto.replace("Ã³", "o").replace("Ãº", "u").replace("Ã±", "n")
    texto = texto.replace("Â", "").replace("Ã", "")

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].apply(limpiar_texto)

# Separar Ceuta y Melilla solo si aparece junto
filas_nuevas = []

for _, row in df.iterrows():
    if row["Comunidad Autonoma"] == "Ceuta y Melilla":
        valor = pd.to_numeric(
            str(row["Residuos"]).replace(".", "").replace(",", "."),
            errors="coerce"
        )
        filas_nuevas.append(["Ceuta", row["Periodo"], valor / 2])
        filas_nuevas.append(["Melilla", row["Periodo"], valor / 2])
    else:
        filas_nuevas.append([row["Comunidad Autonoma"], row["Periodo"], row["Residuos"]])

df = pd.DataFrame(filas_nuevas, columns=["Comunidad Autonoma", "Periodo", "Residuos"])

# Convertir variables a numerico
df["Periodo"] = pd.to_numeric(df["Periodo"], errors="coerce")

df["Residuos"] = (
    df["Residuos"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["Residuos"] = pd.to_numeric(df["Residuos"], errors="coerce")

# Eliminar filas vacias
df = df.dropna(subset=["Comunidad Autonoma", "Periodo", "Residuos"])

In [10]:
ruta_salida = "/content/drive/MyDrive/DATOS_TFG_LIMPIOS/Residuos.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

## **ÍNDICE CALIDAD VIDA**

Se realizó la limpieza del dataset de calidad de vida renombrando las columnas y homogeneizando los nombres de las comunidades autónomas para hacerlos consistentes con el resto de datasets. Se eliminaron tildes, se corrigieron posibles problemas de codificación y se transformó Total Nacional en Espana. Además, se ajustaron denominaciones como Asturias, Principado de a Principado de Asturias o Balears, Illes a Islas Baleares. Finalmente, se convirtieron las variables Periodo y Calidad_Vida a formato numérico para facilitar su análisis.

In [11]:
import pandas as pd
import unicodedata
import re

ruta = "/content/drive/MyDrive/DATOS_TFG/Calidad_Vida_Global.xlsx"

df = pd.read_excel(ruta)

# Renombrar columnas
df.columns = ["Comunidad Autonoma", "Periodo", "Calidad_Vida"]

# Limpiar espacios
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].astype(str).str.strip()

# Funcion para quitar tildes y limpiar texto
def limpiar_texto(texto):
    texto = str(texto)

    # Arreglar caracteres raros frecuentes
    texto = texto.replace("Ã¡", "a").replace("Ã©", "e").replace("Ã­", "i")
    texto = texto.replace("Ã³", "o").replace("Ãº", "u").replace("Ã±", "n")
    texto = texto.replace("Â", "").replace("Ã", "")

    # Quitar tildes
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    # Limpiar espacios extra
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

# Aplicar limpieza de texto
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].apply(limpiar_texto)

# Reemplazar nombres para homogeneizar
reemplazos = {
    "Total Nacional": "Espana",
    "Asturias, Principado de": "Principado de Asturias",
    "Balears, Illes": "Islas Baleares",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia, Region de": "Region de Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja, La": "La Rioja",
    "Castilla y Leon": "Castilla y Leon",
    "Castilla - La Mancha": "Castilla-La Mancha",
    "Cataluna": "Cataluna",
    "Pais Vasco": "Pais Vasco",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Valenciana, Comunidad": "Comunidad Valenciana"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos)

# Separar Ceuta y Melilla solo si aparecen juntas
filas_nuevas = []

for _, row in df.iterrows():
    if row["Comunidad Autonoma"] == "Ceuta y Melilla":
        valor = pd.to_numeric(str(row["Calidad_Vida"]).replace(",", "."), errors="coerce")

        filas_nuevas.append(["Ceuta", row["Periodo"], valor])
        filas_nuevas.append(["Melilla", row["Periodo"], valor])
    else:
        filas_nuevas.append([
            row["Comunidad Autonoma"],
            row["Periodo"],
            row["Calidad_Vida"]
        ])

df = pd.DataFrame(filas_nuevas, columns=["Comunidad Autonoma", "Periodo", "Calidad_Vida"])

# Convertir columnas a numerico
df["Periodo"] = pd.to_numeric(df["Periodo"], errors="coerce")

df["Calidad_Vida"] = (
    df["Calidad_Vida"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df["Calidad_Vida"] = pd.to_numeric(df["Calidad_Vida"], errors="coerce")

# Eliminar filas vacias
df = df.dropna(subset=["Comunidad Autonoma", "Periodo", "Calidad_Vida"])

In [12]:
ruta_salida = "/content/drive/MyDrive/DATOS_TFG_LIMPIOS/Calidad_Vida_Global.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

## **CIFRAS DE POBLACIÓN**
Se realizó la limpieza del dataset de población renombrando las columnas y homogeneizando los nombres de las comunidades autónomas para mantener la misma estructura que en el resto de datasets. Se eliminaron los códigos numéricos iniciales, se corrigieron problemas de codificación y se eliminaron las tildes para asegurar la consistencia textual. Además, se transformó Total Nacional en Espana y se ajustaron denominaciones como Asturias, Principado de o Balears, Illes a un formato homogéneo. Finalmente, las variables Periodo y Poblacion se convirtieron a formato numérico para facilitar su análisis.

In [13]:
import pandas as pd
import unicodedata
import re

ruta = "/content/drive/MyDrive/DATOS_TFG/Cifras_Poblacion.xlsx"

df = pd.read_excel(ruta)

# Renombrar columnas
df.columns = ["Comunidad Autonoma", "Periodo", "Poblacion"]

# Limpiar espacios
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].astype(str).str.strip()

# Quitar codigos numericos iniciales
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].str.replace(r"^\d+\s*", "", regex=True)

# Funcion para limpiar texto, quitar tildes y corregir caracteres raros
def limpiar_texto(texto):
    texto = str(texto)

    texto = texto.replace("Ã¡", "a").replace("Ã©", "e").replace("Ã­", "i")
    texto = texto.replace("Ã³", "o").replace("Ãº", "u").replace("Ã±", "n")
    texto = texto.replace("Â", "").replace("Ã", "")

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].apply(limpiar_texto)

# Homogeneizar nombres
reemplazos = {
    "Total Nacional": "Espana",
    "Asturias, Principado de": "Principado de Asturias",
    "Balears, Illes": "Islas Baleares",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia, Region de": "Region de Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja, La": "La Rioja",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Valenciana, Comunidad": "Comunidad Valenciana",
    "Castilla - La Mancha": "Castilla-La Mancha"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos)

# Separar Ceuta y Melilla solo si aparecen juntas
filas_nuevas = []

for _, row in df.iterrows():
    if row["Comunidad Autonoma"] == "Ceuta y Melilla":
        valor = pd.to_numeric(row["Poblacion"], errors="coerce")

        filas_nuevas.append(["Ceuta", row["Periodo"], valor / 2])
        filas_nuevas.append(["Melilla", row["Periodo"], valor / 2])
    else:
        filas_nuevas.append([
            row["Comunidad Autonoma"],
            row["Periodo"],
            row["Poblacion"]
        ])

df = pd.DataFrame(filas_nuevas, columns=["Comunidad Autonoma", "Periodo", "Poblacion"])

# Convertir a numerico
df["Periodo"] = pd.to_numeric(df["Periodo"], errors="coerce")
df["Poblacion"] = pd.to_numeric(df["Poblacion"], errors="coerce")

# Eliminar filas vacias
df = df.dropna(subset=["Comunidad Autonoma", "Periodo", "Poblacion"])

In [14]:
ruta_salida = "/content/drive/MyDrive/DATOS_TFG_LIMPIOS/Cifras_Poblacion.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

## **DEFUNCIONES**
Se realizó la limpieza del dataset de defunciones renombrando las columnas y homogeneizando los nombres de las comunidades autónomas para mantener la misma estructura que el resto de datasets. Se eliminaron los códigos numéricos iniciales, se corrigieron problemas de codificación y se eliminaron las tildes para asegurar la consistencia textual. Además, se transformó Total Nacional en Espana y se ajustaron denominaciones como Asturias, Principado de o Balears, Illes a un formato homogéneo. Finalmente, las variables Periodo y Defunciones se convirtieron a formato numérico para facilitar su análisis.

In [15]:
import pandas as pd
import unicodedata
import re

ruta = "/content/drive/MyDrive/DATOS_TFG/Defunciones.xlsx"

df = pd.read_excel(ruta)

# Renombrar columnas
df.columns = ["Comunidad Autonoma", "Periodo", "Defunciones"]

# Limpiar espacios
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].astype(str).str.strip()

# Quitar codigos numericos iniciales (ej: 01 Andalucia)
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].str.replace(r"^\d+\s*", "", regex=True)

# Funcion para limpiar tildes y caracteres raros
def limpiar_texto(texto):

    texto = str(texto)

    texto = texto.replace("Ã¡","a").replace("Ã©","e").replace("Ã­","i")
    texto = texto.replace("Ã³","o").replace("Ãº","u").replace("Ã±","n")
    texto = texto.replace("Â","").replace("Ã","")

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].apply(limpiar_texto)

# Homogeneizar nombres
reemplazos = {
    "Nacional": "Espana",
    "Asturias, Principado de": "Principado de Asturias",
    "Balears, Illes": "Islas Baleares",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia, Region de": "Region de Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja, La": "La Rioja",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Valenciana, Comunidad": "Comunidad Valenciana",
    "Castilla - La Mancha": "Castilla-La Mancha"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos)

# Separar Ceuta y Melilla solo si aparece junto
filas_nuevas = []

for _, row in df.iterrows():

    if row["Comunidad Autonoma"] == "Ceuta y Melilla":

        valor = pd.to_numeric(str(row["Defunciones"]).replace(".", "").replace(",", "."), errors="coerce")

        filas_nuevas.append(["Ceuta", row["Periodo"], valor / 2])
        filas_nuevas.append(["Melilla", row["Periodo"], valor / 2])

    else:

        filas_nuevas.append([row["Comunidad Autonoma"], row["Periodo"], row["Defunciones"]])

df = pd.DataFrame(filas_nuevas, columns=["Comunidad Autonoma", "Periodo", "Defunciones"])

# Convertir variables a numerico
df["Periodo"] = pd.to_numeric(df["Periodo"], errors="coerce")

df["Defunciones"] = (
    df["Defunciones"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["Defunciones"] = pd.to_numeric(df["Defunciones"], errors="coerce")

# Eliminar filas vacias
df = df.dropna(subset=["Comunidad Autonoma", "Periodo", "Defunciones"])

In [16]:
ruta_salida = "/content/drive/MyDrive/DATOS_TFG_LIMPIOS/Defunciones.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

## **DESIGUALDAD DISTRIBUCIÓN INGRESOS**
Se realizó la limpieza del dataset de distribución de ingresos renombrando las columnas y homogeneizando los nombres de las comunidades autónomas para mantener la misma estructura que el resto de datasets. Se eliminaron los códigos numéricos iniciales, se corrigieron problemas de codificación y se eliminaron las tildes para asegurar la consistencia textual. Además, se transformó Total Nacional en Espana y se ajustaron denominaciones como Asturias, Principado de o Balears, Illes a un formato homogéneo. Finalmente, las variables Periodo y Distribucion_Ingresos se convirtieron a formato numérico para facilitar su análisis.

In [17]:
import pandas as pd
import unicodedata
import re

ruta = "/content/drive/MyDrive/DATOS_TFG/DistribucionIngresos.xlsx"

df = pd.read_excel(ruta)

# Renombrar columnas
df.columns = ["Comunidad Autonoma", "Periodo", "Distribucion_Ingresos"]

# Limpiar espacios
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].astype(str).str.strip()

# Quitar codigos numericos iniciales si los hubiera
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].str.replace(r"^\d+\s*", "", regex=True)

# Funcion para limpiar texto, quitar tildes y corregir caracteres raros
def limpiar_texto(texto):
    texto = str(texto)

    texto = texto.replace("Ã¡", "a").replace("Ã©", "e").replace("Ã­", "i")
    texto = texto.replace("Ã³", "o").replace("Ãº", "u").replace("Ã±", "n")
    texto = texto.replace("Â", "").replace("Ã", "")

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].apply(limpiar_texto)

# Homogeneizar nombres
reemplazos = {
    "Total Nacional": "Espana",
    "Asturias, Principado de": "Principado de Asturias",
    "Balears, Illes": "Islas Baleares",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia, Region de": "Region de Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja, La": "La Rioja",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Valenciana, Comunidad": "Comunidad Valenciana",
    "Castilla - La Mancha": "Castilla-La Mancha"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos)

# Separar Ceuta y Melilla solo si aparece junto
filas_nuevas = []

for _, row in df.iterrows():
    if row["Comunidad Autonoma"] == "Ceuta y Melilla":
        valor = pd.to_numeric(
            str(row["Distribucion_Ingresos"]).replace(".", "").replace(",", "."),
            errors="coerce"
        )

        filas_nuevas.append(["Ceuta", row["Periodo"], valor / 2])
        filas_nuevas.append(["Melilla", row["Periodo"], valor / 2])
    else:
        filas_nuevas.append([
            row["Comunidad Autonoma"],
            row["Periodo"],
            row["Distribucion_Ingresos"]
        ])

df = pd.DataFrame(filas_nuevas, columns=["Comunidad Autonoma", "Periodo", "Distribucion_Ingresos"])

# Convertir variables a numerico
df["Periodo"] = pd.to_numeric(df["Periodo"], errors="coerce")

df["Distribucion_Ingresos"] = (
    df["Distribucion_Ingresos"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["Distribucion_Ingresos"] = pd.to_numeric(df["Distribucion_Ingresos"], errors="coerce")

# Eliminar filas vacias
df = df.dropna(subset=["Comunidad Autonoma", "Periodo", "Distribucion_Ingresos"])

In [18]:
ruta_salida = "/content/drive/MyDrive/DATOS_TFG_LIMPIOS/DesigualdadDistribucionIngresos.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

## **EMISIONES GEI**
Se realizó la limpieza del dataset de emisiones de gases de efecto invernadero renombrando las columnas y homogeneizando los nombres de las comunidades autónomas para mantener la misma estructura que el resto de datasets. Se eliminaron códigos numéricos iniciales, se corrigieron problemas de codificación y se eliminaron las tildes para asegurar la consistencia textual. Además, se transformó Total Nacional en Espana y se ajustaron denominaciones como Asturias, Principado de o Balears, Illes a un formato homogéneo. Finalmente, las variables Periodo y Emisiones_GEI se convirtieron a formato numérico para facilitar su análisis.

In [19]:
import pandas as pd
import unicodedata
import re

ruta = "/content/drive/MyDrive/DATOS_TFG/EmisionesGEI.xlsx"

df = pd.read_excel(ruta)

# Renombrar columnas
df.columns = ["Comunidad Autonoma", "Periodo", "Emisiones_GEI"]

# Limpiar espacios
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].astype(str).str.strip()

# Quitar codigos numericos iniciales si existen
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].str.replace(r"^\d+\s*", "", regex=True)

# Funcion para limpiar texto y quitar tildes
def limpiar_texto(texto):

    texto = str(texto)

    texto = texto.replace("Ã¡","a").replace("Ã©","e").replace("Ã­","i")
    texto = texto.replace("Ã³","o").replace("Ãº","u").replace("Ã±","n")
    texto = texto.replace("Â","").replace("Ã","")

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].apply(limpiar_texto)

# Homogeneizar nombres de comunidades
reemplazos = {
    "Total Nacional": "Espana",
    "Asturias, Principado de": "Principado de Asturias",
    "Balears, Illes": "Islas Baleares",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia, Region de": "Region de Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja, La": "La Rioja",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Valenciana, Comunidad": "Comunidad Valenciana",
    "Castilla - La Mancha": "Castilla-La Mancha"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos)

# Separar Ceuta y Melilla solo si aparecen juntas
filas_nuevas = []

for _, row in df.iterrows():

    if row["Comunidad Autonoma"] == "Ceuta y Melilla":

        valor = pd.to_numeric(
            str(row["Emisiones_GEI"]).replace(".", "").replace(",", "."),
            errors="coerce"
        )

        filas_nuevas.append(["Ceuta", row["Periodo"], valor / 2])
        filas_nuevas.append(["Melilla", row["Periodo"], valor / 2])

    else:

        filas_nuevas.append([
            row["Comunidad Autonoma"],
            row["Periodo"],
            row["Emisiones_GEI"]
        ])

df = pd.DataFrame(filas_nuevas, columns=["Comunidad Autonoma", "Periodo", "Emisiones_GEI"])

# Convertir variables a numerico
df["Periodo"] = pd.to_numeric(df["Periodo"], errors="coerce")

df["Emisiones_GEI"] = (
    df["Emisiones_GEI"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["Emisiones_GEI"] = pd.to_numeric(df["Emisiones_GEI"], errors="coerce")

# Eliminar filas vacias
df = df.dropna(subset=["Comunidad Autonoma", "Periodo", "Emisiones_GEI"])

In [20]:
ruta_salida = "/content/drive/MyDrive/DATOS_TFG_LIMPIOS/EmisionesGEI.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

## **ÍNDICE CIFRAS NEGOCIO**
Se realizó la limpieza del dataset del índice de cifras de negocio renombrando las columnas para mantener la misma estructura que el resto de datasets utilizados en el análisis. Se eliminaron posibles códigos numéricos iniciales en los nombres de las comunidades autónomas, se corrigieron problemas de codificación y se eliminaron las tildes para asegurar la consistencia textual. Además, se homogeneizaron denominaciones como Asturias, Principado de o Balears, Illes para que coincidieran con el formato utilizado en los demás conjuntos de datos. Finalmente, las variables Periodo e Indice_Cifras_Negocio se transformaron a formato numérico para facilitar su análisis.

In [21]:
import pandas as pd
import unicodedata
import re

ruta = "/content/drive/MyDrive/DATOS_TFG/IndiceCifrasNegocio.xlsx"

df = pd.read_excel(ruta)

# Renombrar columnas segun el numero de columnas
if len(df.columns) == 3:
    df.columns = ["Comunidad Autonoma", "Periodo", "Indice_Cifras_Negocio"]
elif len(df.columns) == 4:
    df.columns = ["Comunidad Autonoma", "Indicador", "Periodo", "Indice_Cifras_Negocio"]

    # Si hay columna indicador, la eliminamos
    df = df.drop(columns=["Indicador"])

# Limpiar espacios
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].astype(str).str.strip()

# Quitar codigos numericos iniciales si existen
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].str.replace(r"^\d+\s*", "", regex=True)

# Funcion para limpiar texto, quitar tildes y corregir caracteres raros
def limpiar_texto(texto):
    texto = str(texto)

    texto = texto.replace("Ã¡", "a").replace("Ã©", "e").replace("Ã­", "i")
    texto = texto.replace("Ã³", "o").replace("Ãº", "u").replace("Ã±", "n")
    texto = texto.replace("Â", "").replace("Ã", "")

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].apply(limpiar_texto)

# Homogeneizar nombres
reemplazos = {
    "Total Nacional": "Espana",
    "Asturias, Principado de": "Principado de Asturias",
    "Balears, Illes": "Islas Baleares",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia, Region de": "Region de Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja, La": "La Rioja",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Valenciana, Comunidad": "Comunidad Valenciana",
    "Castilla - La Mancha": "Castilla-La Mancha"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos)

# Separar Ceuta y Melilla solo si aparecen juntas
filas_nuevas = []

for _, row in df.iterrows():
    if row["Comunidad Autonoma"] == "Ceuta y Melilla":
        valor = pd.to_numeric(
            str(row["Indice_Cifras_Negocio"]).replace(".", "").replace(",", "."),
            errors="coerce"
        )

        filas_nuevas.append(["Ceuta", row["Periodo"], valor / 2])
        filas_nuevas.append(["Melilla", row["Periodo"], valor / 2])
    else:
        filas_nuevas.append([
            row["Comunidad Autonoma"],
            row["Periodo"],
            row["Indice_Cifras_Negocio"]
        ])

df = pd.DataFrame(
    filas_nuevas,
    columns=["Comunidad Autonoma", "Periodo", "Indice_Cifras_Negocio"]
)

# Convertir variables a numerico
df["Periodo"] = pd.to_numeric(df["Periodo"], errors="coerce")

df["Indice_Cifras_Negocio"] = (
    df["Indice_Cifras_Negocio"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["Indice_Cifras_Negocio"] = pd.to_numeric(df["Indice_Cifras_Negocio"], errors="coerce")

# Eliminar filas vacias
df = df.dropna(subset=["Comunidad Autonoma", "Periodo", "Indice_Cifras_Negocio"])

In [22]:
ruta_salida = "/content/drive/MyDrive/DATOS_TFG_LIMPIOS/IndiceCifrasNegocio.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

## **ÍNDICE PRODUCCIÓN INDUSTRIAL**











In [23]:
import pandas as pd
import unicodedata
import re

ruta = "/content/drive/MyDrive/DATOS_TFG/IndiceProduccionIndustrial.xlsx"

df = pd.read_excel(ruta)

# Renombrar columnas segun el numero de columnas
if len(df.columns) == 3:
    df.columns = ["Comunidad Autonoma", "Periodo", "Indice_Produccion_Industrial"]
elif len(df.columns) == 4:
    df.columns = ["Comunidad Autonoma", "Indicador", "Periodo", "Indice_Produccion_Industrial"]

    # Si hay columna indicador, la eliminamos
    df = df.drop(columns=["Indicador"])

# Limpiar espacios
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].astype(str).str.strip()

# Quitar codigos numericos iniciales si existen
df["Comunidad Autonoma"] = df["Comunidad Autonoma"].str.replace(r"^\d+\s*", "", regex=True)

# Funcion para limpiar texto, quitar tildes y corregir caracteres raros
def limpiar_texto(texto):
    texto = str(texto)

    texto = texto.replace("Ã¡", "a").replace("Ã©", "e").replace("Ã­", "i")
    texto = texto.replace("Ã³", "o").replace("Ãº", "u").replace("Ã±", "n")
    texto = texto.replace("Â", "").replace("Ã", "")

    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].apply(limpiar_texto)

# Homogeneizar nombres
reemplazos = {
    "Total Nacional": "Espana",
    "Asturias, Principado de": "Principado de Asturias",
    "Balears, Illes": "Islas Baleares",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia, Region de": "Region de Murcia",
    "Navarra, Comunidad Foral de": "Navarra",
    "Rioja, La": "La Rioja",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Valenciana, Comunidad": "Comunidad Valenciana",
    "Castilla - La Mancha": "Castilla-La Mancha"
}

df["Comunidad Autonoma"] = df["Comunidad Autonoma"].replace(reemplazos)

# Separar Ceuta y Melilla solo si aparecen juntas
filas_nuevas = []

for _, row in df.iterrows():
    if row["Comunidad Autonoma"] == "Ceuta y Melilla":
        valor = pd.to_numeric(
            str(row["Indice_Produccion_Industrial"]).replace(".", "").replace(",", "."),
            errors="coerce"
        )

        filas_nuevas.append(["Ceuta", row["Periodo"], valor / 2])
        filas_nuevas.append(["Melilla", row["Periodo"], valor / 2])
    else:
        filas_nuevas.append([
            row["Comunidad Autonoma"],
            row["Periodo"],
            row["Indice_Produccion_Industrial"]
        ])

df = pd.DataFrame(
    filas_nuevas,
    columns=["Comunidad Autonoma", "Periodo", "Indice_Produccion_Industrial"]
)

# Convertir variables a numerico
df["Periodo"] = pd.to_numeric(df["Periodo"], errors="coerce")

df["Indice_Produccion_Industrial"] = (
    df["Indice_Produccion_Industrial"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df["Indice_Produccion_Industrial"] = pd.to_numeric(df["Indice_Produccion_Industrial"], errors="coerce")

# Eliminar filas vacias
df = df.dropna(subset=["Comunidad Autonoma", "Periodo", "Indice_Produccion_Industrial"])

In [24]:
ruta_salida = "/content/drive/MyDrive/DATOS_TFG_LIMPIOS/IndiceProduccionIndustrial.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")